# INSE 6450 – AI in Systems Engineering## Milestone 4 – Adaptive Study Buddy### Continual Learning, Human-in-the-Loop & Active Learning**Name:** Sohana Mahmud  **Student ID:** 40271073  **Winter 2026**

---# Setup: Data Loading, Feature Engineering & Model TrainingWe reuse the pipeline from Milestone 3 with all permanent improvements:- **StandardScaler** normalization- **Early stopping** with patience=5- **Label smoothing** (α=0.05)- **Feature jittering** augmentation- **Temperature scaling** (T=1.014)

In [ ]:
#!/usr/bin/env python3"""INSE 6450 – AI in Systems EngineeringMilestone 4 – Adaptive Study BuddyContinual Learning, Human-in-the-Loop & Active LearningName: Sohana MahmudStudent ID: 40271073Winter 2026This script implements the full M4 pipeline building on M3:1. Continual Learning with Experience Replay (reservoir buffer)2. Human-in-the-Loop: drift gating + label correction3. Active Learning: hybrid uncertainty+diversity sampling4. Full system integration with plots and metrics"""import numpy as npimport pandas as pdimport torchimport torch.nn as nnimport torch.nn.functional as Fimport time, os, copy, warnings, hashlib, datetime, json, randomimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport matplotlib.gridspec as gridspecmatplotlib.rcParams['figure.dpi'] = 120warnings.filterwarnings("ignore")from sklearn.model_selection import GroupShuffleSplitfrom sklearn.metrics import (accuracy_score, precision_score, recall_score,                             f1_score, roc_auc_score, average_precision_score,                             roc_curve, precision_recall_curve)from sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansfrom torch.utils.data import DataLoader, TensorDataset# CONFIGURATION (consistent with M2-M3)

## Data Loading & Feature Engineering

In [ ]:
CONFIG = {    "seed": 42,    "batch_size": 2048,    "learning_rate": 1e-3,    "finetune_lr": 5e-4,    "epochs": 30,    "patience": 5,    "label_smoothing": 0.05,    "temperature": 1.014,    # M4-specific    "reservoir_size": 10000,    "replay_mix_ratio": 0.5,    # 50% old / 50% new    "al_query_size": 100,       # samples per AL cycle    "al_preselect_k": 300,      # uncertainty pre-selection pool    "al_cycles": 5,    "drift_psi_threshold": 0.2,    "freeze_layers": 6,}torch.manual_seed(CONFIG["seed"])np.random.seed(CONFIG["seed"])random.seed(CONFIG["seed"])device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Device: {device}")print(f"Config: {json.dumps(CONFIG, indent=2)}")# DATA LOADING & FEATURE ENGINEERING (from M1-M3)

## Model Definition (MLP from M2-M3)

In [ ]:
print("\n" + "="*70)print("PHASE 0: Data Loading & Feature Engineering (M1-M3 pipeline)")print("="*70)df = pd.read_csv("data.csv")df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)print(f"Dataset: {len(df):,} rows, {df['user_id'].nunique():,} users, {df['content_id'].nunique():,} items")print(f"Class balance: {df['answered_correctly'].mean():.3f} positive rate")# Feature EngineeringWIN = 20q_stats = df.groupby("content_id")["answered_correctly"].agg(["mean", "count"])q_stats.columns = ["q_correct_rate", "q_count"]df = df.join(q_stats, on="content_id")g = df.groupby("user_id", sort=False)past = g["answered_correctly"].shift(1)df["u_prev_correct"] = past.fillna(0)df["u_accuracy"] = (    past.groupby(df["user_id"])        .rolling(WIN, min_periods=5)        .mean()        .reset_index(level=0, drop=True))df["u_accuracy"] = df["u_accuracy"].fillna(df["u_accuracy"].mean())df["u_attempts"] = g.cumcount()df["u_delta"] = g["timestamp"].diff().fillna(0)df["u_log_delta"] = np.log1p(df["u_delta"].clip(lower=0))df["log_elapsed"] = np.log1p(df["prior_question_elapsed_time"].clip(lower=0))df["prior_expl"] = df["prior_question_had_explanation"].fillna(False).astype(bool).astype(int)FEATURES = [    "q_correct_rate", "q_count",    "u_prev_correct", "u_accuracy", "u_attempts",    "u_log_delta", "log_elapsed", "prior_expl"]X = df[FEATURES].values.astype("float32")y = df["answered_correctly"].values.astype("float32")# Train/Val/Test split (user-level)scaler = StandardScaler()splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)train_idx, temp_idx = next(splitter.split(X, y, groups=df["user_id"]))X_train_raw, y_train = X[train_idx], y[train_idx]X_temp_raw, y_temp = X[temp_idx], y[temp_idx]splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)val_idx, test_idx = next(splitter2.split(X_temp_raw, y_temp, groups=df["user_id"].iloc[temp_idx]))X_val_raw, y_val = X_temp_raw[val_idx], y_temp[val_idx]X_test_raw, y_test = X_temp_raw[test_idx], y_temp[test_idx]scaler.fit(X_train_raw)X_train = scaler.transform(X_train_raw).astype("float32")X_val = scaler.transform(X_val_raw).astype("float32")X_test = scaler.transform(X_test_raw).astype("float32")train_mean = X_train.mean(axis=0)train_std = X_train.std(axis=0)print(f"  Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}")

## Train Baseline Model

In [ ]:
# MODEL DEFINITION (from M2-M3)

## PSI Computation Utility (from M3)

In [ ]:
class MLP(nn.Module):    def __init__(self, input_dim):        super().__init__()        self.net = nn.Sequential(            nn.Linear(input_dim, 128),   # layer 0,1            nn.ReLU(),                    # layer 2            nn.BatchNorm1d(128),          # layer 3            nn.Dropout(0.3),              # layer 4            nn.Linear(128, 64),           # layer 5            nn.ReLU(),                    # layer 6            nn.BatchNorm1d(64),           # layer 7            nn.Dropout(0.2),              # layer 8            nn.Linear(64, 32),            # layer 9            nn.ReLU(),                    # layer 10            nn.Linear(32, 1)              # layer 11        )    def forward(self, x):        return self.net(x).squeeze(1)def train_model(model, X_tr, y_tr, X_v, y_v, config, label_smoothing=0.0, verbose=True):    pos_weight = torch.tensor([(len(y_tr) - y_tr.sum()) / y_tr.sum()]).to(device)    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)    best_auc = 0; best_state = None; patience_counter = 0    train_losses = []; val_aucs = []    for epoch in range(config["epochs"]):        model.train()        epoch_loss = 0        for xb, yb in train_loader:            xb, yb = xb.to(device), yb.to(device)            if label_smoothing > 0:                yb = yb * (1 - label_smoothing) + 0.5 * label_smoothing            optimizer.zero_grad()            loss = criterion(model(xb), yb)            loss.backward()            optimizer.step()            epoch_loss += loss.item()        train_losses.append(epoch_loss / len(train_loader))        model.eval()        with torch.no_grad():            preds = torch.sigmoid(model(torch.from_numpy(X_v).to(device))).cpu().numpy()        auc = roc_auc_score(y_v, preds)        val_aucs.append(auc)        if auc > best_auc:            best_auc = auc            best_state = copy.deepcopy(model.state_dict())            patience_counter = 0        else:            patience_counter += 1            if patience_counter >= config["patience"]:                if verbose:                    print(f"  Early stop at epoch {epoch+1}, best val AUC: {best_auc:.4f}")                break    model.load_state_dict(best_state)    return model, train_losses, val_aucsdef evaluate(model, X_data, y_data, threshold=0.5):    model.eval()    with torch.no_grad():        preds = torch.sigmoid(model(torch.from_numpy(X_data).to(device))).cpu().numpy()    labels = (preds > threshold).astype(int)    return {        "Accuracy": accuracy_score(y_data, labels),        "Precision": precision_score(y_data, labels, zero_division=0),        "Recall": recall_score(y_data, labels, zero_division=0),        "F1": f1_score(y_data, labels, zero_division=0),        "AUROC": roc_auc_score(y_data, preds),        "PR-AUC": average_precision_score(y_data, preds)    }, predsdef measure_latency(model, X_data, n_runs=1000):    model.eval()    sample = torch.from_numpy(X_data[:1]).to(device)    # Warmup    for _ in range(100):        with torch.no_grad():            model(sample)    times = []    for _ in range(n_runs):        t0 = time.perf_counter()        with torch.no_grad():            model(sample)        times.append((time.perf_counter() - t0) * 1000)    return {"p50_ms": np.percentile(times, 50), "p90_ms": np.percentile(times, 90),            "throughput_sps": 1000 / np.mean(times)}

---# 1. Continual Learning Strategy (3.5 pts)## 1.1 Three CL Strategies| Strategy | Fit | Reason ||---|---|---|| EWC | Moderate | Requires Fisher matrix; assumes task boundaries || **Experience Replay** | **Strong** | Natural for gradual drift; validated in M3 || Progressive Nets | Poor | Designed for distinct tasks; unnecessary growth |

## 1.2 Reservoir Buffer & ER Implementation

In [ ]:
# TRAIN BASELINE MODEL (M3 pipeline)

## 1.3 Incremental Fine-tuning with Experience Replay

In [ ]:
print("\n" + "="*70)print("PHASE 0.5: Train Baseline MLP (M3 config)")print("="*70)t0 = time.time()model = MLP(len(FEATURES)).to(device)model, losses, aucs = train_model(model, X_train, y_train, X_val, y_val, CONFIG,                                   label_smoothing=CONFIG["label_smoothing"])baseline_train_time = time.time() - t0clean_metrics, clean_preds = evaluate(model, X_test, y_test)print("\n=== Clean Test Metrics (Baseline M3) ===")for k, v in clean_metrics.items():    print(f"  {k}: {v:.4f}")print(f"  Training time: {baseline_train_time:.1f}s")baseline_latency = measure_latency(model, X_test)print(f"  p50 latency: {baseline_latency['p50_ms']:.3f} ms")print(f"  p90 latency: {baseline_latency['p90_ms']:.3f} ms")print(f"  Throughput: {baseline_latency['throughput_sps']:,.0f} sps")# Save baseline statebaseline_state = copy.deepcopy(model.state_dict())

## 1.4 Multi-Step Drift Simulation

In [ ]:
# PSI COMPUTATION UTILITY (from M3)

---# 2. Human-in-the-Loop Learning (3.5 pts)## 2.1 When & Why Humans Intervene- **(a)** Label low-confidence predictions [0.45, 0.55]- **(b)** Validate drift alerts (gate before CL updates)- **(c)** Override cold-start question difficulty## 2.2 How Human Input is Incorporated- As **new labeled data** (2× weight, priority buffer insertion)- As **gating logic** (approve/reject drift alerts)

## 2.3 Active Learning Strategy (Hybrid Uncertainty + Diversity)

In [ ]:
def compute_psi(reference, current, bins=10):    """Population Stability Index between two distributions."""    breakpoints = np.percentile(reference, np.linspace(0, 100, bins + 1))    breakpoints[0] = -np.inf    breakpoints[-1] = np.inf    ref_counts = np.histogram(reference, bins=breakpoints)[0] / len(reference)    cur_counts = np.histogram(current, bins=breakpoints)[0] / len(current)    ref_counts = np.clip(ref_counts, 1e-6, None)    cur_counts = np.clip(cur_counts, 1e-6, None)    psi = np.sum((cur_counts - ref_counts) * np.log(cur_counts / ref_counts))    return psi

## 2.4 Active Learning & HITL Simulation

In [ ]:
## SECTION 1: CONTINUAL LEARNING STRATEGY (3.5 pts)#

---# 3. Integration into Complete AI System (3.0 pts)## 3.1 System Architecture Diagram

In [ ]:
print("\n" + "="*70)print("SECTION 1: CONTINUAL LEARNING WITH EXPERIENCE REPLAY")print("="*70)# ─── 1.1 Discussion of 3 CL strategies (see report) ───print("\n--- 1.1 Three CL Strategies Discussed ---")print("1. Elastic Weight Consolidation (EWC): Moderate fit")print("2. Experience Replay (ER) with Reservoir: SELECTED — Strong fit")print("3. Progressive Neural Networks: Poor fit")# ─── 1.2 Reservoir Buffer Implementation ───print("\n--- 1.2 Reservoir Buffer Implementation ---")class ReservoirBuffer:    """Reservoir sampling buffer (Vitter's Algorithm R)."""    def __init__(self, capacity, feature_dim):        self.capacity = capacity        self.X = np.zeros((capacity, feature_dim), dtype="float32")        self.y = np.zeros(capacity, dtype="float32")        self.count = 0  # total samples seen        self.size = 0   # current buffer occupancy    def add_batch(self, X_new, y_new):        """Add samples using reservoir sampling."""        for i in range(len(X_new)):            if self.size < self.capacity:                self.X[self.size] = X_new[i]                self.y[self.size] = y_new[i]                self.size += 1            else:                j = np.random.randint(0, self.count + 1)                if j < self.capacity:                    self.X[j] = X_new[i]                    self.y[j] = y_new[i]            self.count += 1    def sample(self, n):        """Sample n items from buffer."""        n = min(n, self.size)        idx = np.random.choice(self.size, n, replace=False)        return self.X[idx], self.y[idx]    def get_all(self):        return self.X[:self.size], self.y[:self.size]    def memory_mb(self):        return (self.X[:self.size].nbytes + self.y[:self.size].nbytes) / (1024**2)# Initialize buffer with training databuffer = ReservoirBuffer(CONFIG["reservoir_size"], len(FEATURES))buffer.add_batch(X_train, y_train)print(f"  Buffer initialized: {buffer.size}/{buffer.capacity} samples")print(f"  Buffer memory: {buffer.memory_mb():.2f} MB")# ─── 1.3 Incremental Fine-tuning with Experience Replay ───def incremental_finetune_er(model, X_new, y_new, buffer, X_val, y_val, config):    """Fine-tune model using Experience Replay: mix buffer + new data."""    t0 = time.time()    # Sample from buffer (50% old, 50% new)    n_new = len(X_new)    n_replay = min(n_new, buffer.size)    X_buf, y_buf = buffer.sample(n_replay)    X_combined = np.concatenate([X_buf, X_new], axis=0)    y_combined = np.concatenate([y_buf, y_new], axis=0)    # Shuffle    perm = np.random.permutation(len(X_combined))    X_combined = X_combined[perm]    y_combined = y_combined[perm]    # Freeze first layers (preserves low-level feature extraction)    freeze_count = config["freeze_layers"]    for i, (name, param) in enumerate(model.named_parameters()):        if i < freeze_count:            param.requires_grad = False        else:            param.requires_grad = True    pos_weight = torch.tensor([(len(y_combined) - y_combined.sum()) / max(y_combined.sum(), 1)]).to(device)    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)    trainable_params = [p for p in model.parameters() if p.requires_grad]    optimizer = torch.optim.Adam(trainable_params, lr=config["finetune_lr"])    train_ds = TensorDataset(torch.from_numpy(X_combined), torch.from_numpy(y_combined))    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)    best_auc = 0; best_state = None; patience_counter = 0    for epoch in range(config["epochs"]):        model.train()        for xb, yb in train_loader:            xb, yb = xb.to(device), yb.to(device)            if config["label_smoothing"] > 0:                yb = yb * (1 - config["label_smoothing"]) + 0.5 * config["label_smoothing"]            optimizer.zero_grad()            loss = criterion(model(xb), yb)            loss.backward()            optimizer.step()        model.eval()        with torch.no_grad():            preds = torch.sigmoid(model(torch.from_numpy(X_val).to(device))).cpu().numpy()        auc = roc_auc_score(y_val, preds)        if auc > best_auc:            best_auc = auc            best_state = copy.deepcopy(model.state_dict())            patience_counter = 0        else:            patience_counter += 1            if patience_counter >= config["patience"]:                break    model.load_state_dict(best_state)    # Unfreeze all layers    for param in model.parameters():        param.requires_grad = True    # Update buffer with new data    buffer.add_batch(X_new, y_new)    update_time = time.time() - t0    return model, update_time# ─── 1.4 Simulated Drift & Continual Update Experiment ───print("\n--- 1.4 Multi-Step Drift Simulation (5 time steps) ---")def generate_drifted_data(X_base, y_base, drift_magnitude, scaler_mean, scaler_std):    """Generate drifted version of test data."""    X_drifted = X_base.copy()    # Apply feature-specific shifts (in standardized space)    shifts = {        0: -0.3 * drift_magnitude,  # q_correct_rate: harder items        3: +0.4 * drift_magnitude,  # u_accuracy: weaker students (shifted mean)        4: -0.3 * drift_magnitude,  # u_attempts: fewer prior interactions        6: +0.2 * drift_magnitude,  # log_elapsed: slower responses        7: +0.16 * drift_magnitude, # prior_expl: more explanation usage    }    for feat_idx, shift in shifts.items():        noise = np.random.normal(0, 0.3 * abs(shift), size=len(X_drifted))        X_drifted[:, feat_idx] += shift + noise    return X_drifted# Generate simulated "new data" for each drift stepdrift_results = []model_cl = copy.deepcopy(model)model_cl.load_state_dict(baseline_state)print(f"\n{'Step':<8} {'AUROC Before':<15} {'AUROC After':<14} {'F1 After':<12} "      f"{'Forgetting':<12} {'Update(s)':<10}")print("-" * 75)# Baselinedrift_results.append({    "step": 0, "auroc_before": clean_metrics["AUROC"], "auroc_after": None,    "f1_after": clean_metrics["F1"], "forgetting": 0, "update_time": 0})print(f"{'T=0':<8} {clean_metrics['AUROC']:<15.4f} {'—':<14} {clean_metrics['F1']:<12.4f} "      f"{'—':<12} {'—':<10}")auroc_before_list = [clean_metrics["AUROC"]]auroc_after_list = [clean_metrics["AUROC"]]forgetting_list = [0]for step in range(1, 6):    # Generate drifted data    X_drifted = generate_drifted_data(X_test, y_test, step, train_mean, train_std)    # Measure BEFORE update    metrics_before, _ = evaluate(model_cl, X_drifted, y_test)    auroc_before = metrics_before["AUROC"]    # Generate "new training data" from drift distribution    n_new = 5000    idx_new = np.random.choice(len(X_train), n_new, replace=False)    X_new_raw = X_train[idx_new].copy()    y_new = y_train[idx_new].copy()    # Apply drift to new data    X_new = generate_drifted_data(X_new_raw, y_new, step, train_mean, train_std)    # Perform ER update    model_cl, update_time = incremental_finetune_er(        model_cl, X_new, y_new, buffer, X_val, y_val, CONFIG    )    # Measure AFTER update    metrics_after, _ = evaluate(model_cl, X_drifted, y_test)    auroc_after = metrics_after["AUROC"]    # Measure forgetting on clean data    clean_after, _ = evaluate(model_cl, X_test, y_test)    forgetting = clean_metrics["AUROC"] - clean_after["AUROC"]    drift_results.append({        "step": step, "auroc_before": auroc_before, "auroc_after": auroc_after,        "f1_after": metrics_after["F1"], "forgetting": forgetting, "update_time": update_time    })    auroc_before_list.append(auroc_before)    auroc_after_list.append(auroc_after)    forgetting_list.append(forgetting)    print(f"{'T='+str(step):<8} {auroc_before:<15.4f} {auroc_after:<14.4f} "          f"{metrics_after['F1']:<12.4f} {forgetting:<12.4f} {update_time:<10.1f}")# ─── Plot: Continual Learning Metric Trajectories ───fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))steps = list(range(6))axes[0].plot(steps, auroc_before_list, 'r-o', label='Before ER Update', linewidth=2)axes[0].plot(steps, auroc_after_list, 'g-s', label='After ER Update', linewidth=2)axes[0].axhline(y=clean_metrics["AUROC"], color='b', linestyle='--', alpha=0.5, label='Clean Baseline')axes[0].set_xlabel("Drift Time Step")axes[0].set_ylabel("AUROC")axes[0].set_title("AUROC: Before vs After ER Update")axes[0].legend(fontsize=8)axes[0].grid(True, alpha=0.3)axes[1].bar(steps, forgetting_list, color=['green' if f < 0.005 else 'orange' for f in forgetting_list])axes[1].axhline(y=0.02, color='r', linestyle='--', label='Rollback Threshold')axes[1].set_xlabel("Drift Time Step")axes[1].set_ylabel("Forgetting (ΔAUROC)")axes[1].set_title("Catastrophic Forgetting Check")axes[1].legend(fontsize=8)axes[1].grid(True, alpha=0.3)update_times = [r["update_time"] for r in drift_results]axes[2].bar(steps, update_times, color='steelblue')axes[2].set_xlabel("Drift Time Step")axes[2].set_ylabel("Update Time (s)")axes[2].set_title("ER Update Efficiency")axes[2].grid(True, alpha=0.3)plt.tight_layout()plt.savefig("m4_continual_learning_trajectories.png", dpi=150, bbox_inches='tight')plt.close()print("\n[SAVED] m4_continual_learning_trajectories.png")# Efficiency metrics during CLprint("\n--- Efficiency Metrics During CL Updates ---")cl_latency = measure_latency(model_cl, X_test)import psutilram_usage = psutil.Process().memory_info().rss / (1024**2)param_count = sum(p.numel() for p in model_cl.parameters())model_size_mb = sum(p.numel() * 4 for p in model_cl.parameters()) / (1024**2)print(f"  Parameters: {param_count:,}")print(f"  Model size: {model_size_mb:.4f} MB")print(f"  Buffer memory: {buffer.memory_mb():.2f} MB")print(f"  p50 latency: {cl_latency['p50_ms']:.3f} ms")print(f"  p90 latency: {cl_latency['p90_ms']:.3f} ms")print(f"  Throughput: {cl_latency['throughput_sps']:,.0f} sps")print(f"  Avg update time: {np.mean([r['update_time'] for r in drift_results[1:]]):.1f} s")print(f"  Process RAM: {ram_usage:.1f} MB")

## 3.2–3.4 Updated Failure Modes, Impact Analysis & Cross-Milestone Summary

In [ ]:
## SECTION 2: HUMAN-IN-THE-LOOP LEARNING (3.5 pts)#

## Final Performance & Artifacts

In [ ]:
print("\n" + "="*70)print("SECTION 2: HUMAN-IN-THE-LOOP & ACTIVE LEARNING")print("="*70)# ─── 2.1 When & Why Humans Intervene ───print("\n--- 2.1 Human Intervention Points ---")print("(a) Label low-confidence predictions [0.45, 0.55]")print("(b) Validate drift alerts (gate before CL updates)")print("(c) Override cold-start question difficulty")# ─── 2.2 How Human Input is Incorporated ───print("\n--- 2.2 Human Input Incorporation ---")print("- As new labeled data (2x weight, priority buffer insertion)")print("- As gating logic (approve/reject drift alerts)")# ─── 2.3 Active Learning Strategy ───print("\n--- 2.3 Active Learning: Hybrid Uncertainty + Diversity ---")def active_learning_query(model, X_pool, query_size, preselect_k):    """Hybrid uncertainty + diversity active learning query."""    model.eval()    with torch.no_grad():        preds = torch.sigmoid(model(torch.from_numpy(X_pool).to(device))).cpu().numpy()    # Stage 1: Uncertainty pre-selection (margin sampling)    margins = np.abs(preds - 0.5)    uncertain_idx = np.argsort(margins)[:preselect_k]    # Stage 2: Diversity filtering via KMeans    X_uncertain = X_pool[uncertain_idx]    kmeans = KMeans(n_clusters=query_size, random_state=42, n_init=3)    kmeans.fit(X_uncertain)    # Select sample closest to each centroid    selected = []    for k in range(query_size):        cluster_members = np.where(kmeans.labels_ == k)[0]        if len(cluster_members) > 0:            dists = np.linalg.norm(X_uncertain[cluster_members] - kmeans.cluster_centers_[k], axis=1)            closest = cluster_members[np.argmin(dists)]            selected.append(uncertain_idx[closest])    return np.array(selected), predsdef simulate_human_oracle(y_true, accuracy=0.95):    """Simulate human annotator with given accuracy."""    labels = y_true.copy()    n_flip = int(len(labels) * (1 - accuracy))    flip_idx = np.random.choice(len(labels), n_flip, replace=False)    labels[flip_idx] = 1 - labels[flip_idx]    return labels# ─── 2.4 Active Learning Simulation ───print("\n--- 2.4 Active Learning & HITL Simulation (5 cycles) ---")# Reset model to baselinemodel_al = MLP(len(FEATURES)).to(device)model_al.load_state_dict(baseline_state)# Create unlabeled pool (simulate by holding out labels)pool_idx = np.random.choice(len(X_train), min(50000, len(X_train)), replace=False)X_pool = X_train[pool_idx].copy()y_pool = y_train[pool_idx].copy()labeled_mask = np.zeros(len(X_pool), dtype=bool)al_results = []al_results.append({    "cycle": 0, "total_labels": 0, "auroc": clean_metrics["AUROC"],    "f1": clean_metrics["F1"], "labeling_pct": 0, "update_time": 0})# Also run random sampling baseline for comparisonmodel_random = MLP(len(FEATURES)).to(device)model_random.load_state_dict(baseline_state)random_results = [{"cycle": 0, "auroc": clean_metrics["AUROC"]}]# Pure uncertainty baselinemodel_unc = MLP(len(FEATURES)).to(device)model_unc.load_state_dict(baseline_state)unc_results = [{"cycle": 0, "auroc": clean_metrics["AUROC"]}]print(f"\n{'Cycle':<8} {'Queries':<10} {'Cumul.':<10} {'AUROC':<10} {'F1':<10} "      f"{'Label%':<10} {'Time(s)':<10}")print("-" * 68)print(f"{'Base':<8} {'—':<10} {'0':<10} {clean_metrics['AUROC']:<10.4f} "      f"{clean_metrics['F1']:<10.4f} {'0%':<10} {'—':<10}")buffer_al = ReservoirBuffer(CONFIG["reservoir_size"], len(FEATURES))buffer_al.add_batch(X_train, y_train)for cycle in range(1, CONFIG["al_cycles"] + 1):    # ─── Active Learning Query (Hybrid) ───    unlabeled_idx = np.where(~labeled_mask)[0]    if len(unlabeled_idx) < CONFIG["al_preselect_k"]:        break    X_unlabeled = X_pool[unlabeled_idx]    query_idx_in_unlabeled, _ = active_learning_query(        model_al, X_unlabeled, CONFIG["al_query_size"], CONFIG["al_preselect_k"]    )    query_idx = unlabeled_idx[query_idx_in_unlabeled]    # Simulate human annotation    y_human = simulate_human_oracle(y_pool[query_idx], accuracy=0.95)    labeled_mask[query_idx] = True    # Fine-tune with human-labeled data (2x weight via duplication)    X_human = X_pool[query_idx]    X_augmented = np.concatenate([X_human, X_human], axis=0)  # 2x weight    y_augmented = np.concatenate([y_human, y_human], axis=0)    t0 = time.time()    model_al, update_time = incremental_finetune_er(        model_al, X_augmented, y_augmented, buffer_al, X_val, y_val, CONFIG    )    update_time = time.time() - t0    metrics_al, _ = evaluate(model_al, X_test, y_test)    total_labels = labeled_mask.sum()    labeling_pct = total_labels / len(y_train) * 100    al_results.append({        "cycle": cycle, "total_labels": int(total_labels),        "auroc": metrics_al["AUROC"], "f1": metrics_al["F1"],        "labeling_pct": labeling_pct, "update_time": update_time    })    print(f"{'C'+str(cycle):<8} {CONFIG['al_query_size']:<10} {int(total_labels):<10} "          f"{metrics_al['AUROC']:<10.4f} {metrics_al['F1']:<10.4f} "          f"{labeling_pct:<10.3f} {update_time:<10.1f}")    # ─── Random Sampling Baseline ───    random_idx = np.random.choice(unlabeled_idx, CONFIG["al_query_size"], replace=False)    y_random = simulate_human_oracle(y_pool[random_idx], accuracy=0.95)    X_random = X_pool[random_idx]    X_rand_aug = np.concatenate([X_random, X_random], axis=0)    y_rand_aug = np.concatenate([y_random, y_random], axis=0)    buffer_rand = ReservoirBuffer(CONFIG["reservoir_size"], len(FEATURES))    buffer_rand.add_batch(X_train, y_train)    model_random, _ = incremental_finetune_er(        model_random, X_rand_aug, y_rand_aug, buffer_rand, X_val, y_val, CONFIG    )    metrics_rand, _ = evaluate(model_random, X_test, y_test)    random_results.append({"cycle": cycle, "auroc": metrics_rand["AUROC"]})    # ─── Uncertainty-only Baseline ───    model_unc.eval()    with torch.no_grad():        preds_unc = torch.sigmoid(model_unc(torch.from_numpy(X_unlabeled).to(device))).cpu().numpy()    unc_margins = np.abs(preds_unc - 0.5)    unc_query = unlabeled_idx[np.argsort(unc_margins)[:CONFIG["al_query_size"]]]    y_unc_human = simulate_human_oracle(y_pool[unc_query], accuracy=0.95)    X_unc_data = X_pool[unc_query]    X_unc_aug = np.concatenate([X_unc_data, X_unc_data], axis=0)    y_unc_aug = np.concatenate([y_unc_human, y_unc_human], axis=0)    buffer_unc = ReservoirBuffer(CONFIG["reservoir_size"], len(FEATURES))    buffer_unc.add_batch(X_train, y_train)    model_unc, _ = incremental_finetune_er(        model_unc, X_unc_aug, y_unc_aug, buffer_unc, X_val, y_val, CONFIG    )    metrics_unc_eval, _ = evaluate(model_unc, X_test, y_test)    unc_results.append({"cycle": cycle, "auroc": metrics_unc_eval["AUROC"]})# Comparison summaryprint("\n--- Sampling Strategy Comparison (after 5 cycles) ---")print(f"  Random:     AUROC = {random_results[-1]['auroc']:.4f}  (Δ = {random_results[-1]['auroc'] - clean_metrics['AUROC']:+.4f})")print(f"  Uncertainty: AUROC = {unc_results[-1]['auroc']:.4f}  (Δ = {unc_results[-1]['auroc'] - clean_metrics['AUROC']:+.4f})")print(f"  Hybrid:     AUROC = {al_results[-1]['auroc']:.4f}  (Δ = {al_results[-1]['auroc'] - clean_metrics['AUROC']:+.4f})")# Compute relative efficiencygain_random = random_results[-1]["auroc"] - clean_metrics["AUROC"]gain_hybrid = al_results[-1]["auroc"] - clean_metrics["AUROC"]gain_unc = unc_results[-1]["auroc"] - clean_metrics["AUROC"]if gain_random > 0:    print(f"  Hybrid efficiency: {gain_hybrid/gain_random:.1f}× vs random")    print(f"  Uncertainty efficiency: {gain_unc/gain_random:.1f}× vs random")# ─── Plot: Active Learning Results ───fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))cycles = [r["cycle"] for r in al_results]al_aurocs = [r["auroc"] for r in al_results]rand_aurocs = [r["auroc"] for r in random_results]unc_aurocs = [r["auroc"] for r in unc_results]axes[0].plot(cycles, al_aurocs, 'g-s', label='Hybrid (ours)', linewidth=2, markersize=7)axes[0].plot(cycles, unc_aurocs, 'b-^', label='Uncertainty only', linewidth=2, markersize=6)axes[0].plot(cycles, rand_aurocs, 'r-o', label='Random', linewidth=2, markersize=6)axes[0].axhline(y=clean_metrics["AUROC"], color='gray', linestyle='--', alpha=0.5, label='Baseline')axes[0].set_xlabel("Active Learning Cycle")axes[0].set_ylabel("AUROC")axes[0].set_title("Active Learning: Strategy Comparison")axes[0].legend(fontsize=9)axes[0].grid(True, alpha=0.3)# Labeling efficiency bar chartstrategies = ['Random', 'Uncertainty', 'Hybrid']gains = [gain_random, gain_unc, gain_hybrid]colors = ['#e74c3c', '#3498db', '#27ae60']axes[1].bar(strategies, [g * 10000 for g in gains], color=colors)axes[1].set_ylabel("AUROC Gain (×10⁴)")axes[1].set_title("AUROC Improvement per Strategy\n(500 labels = 0.24% of training data)")axes[1].grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.savefig("m4_active_learning_comparison.png", dpi=150, bbox_inches='tight')plt.close()print("\n[SAVED] m4_active_learning_comparison.png")# ─── Active Learning Workflow Diagram ───fig, ax = plt.subplots(1, 1, figsize=(12, 5))ax.set_xlim(0, 12)ax.set_ylim(0, 5)ax.axis('off')ax.set_title("Figure: Active Learning Workflow Diagram", fontsize=14, fontweight='bold', pad=20)boxes = [    (0.5, 3.5, "Unlabeled\nPool U", "#3498db"),    (2.8, 3.5, "Uncertainty\nPre-select\n(Top-K)", "#e67e22"),    (5.3, 3.5, "Diversity\nFiltering\n(K-Medoids)", "#9b59b6"),    (7.8, 3.5, "Human\nAnnotator\n(Oracle)", "#e74c3c"),    (10.0, 3.5, "Labeled\nBatch B", "#27ae60"),    (5.3, 1.0, "ER Update\n(Buffer + New)", "#2c3e50"),    (8.5, 1.0, "Updated\nModel v(t+1)", "#16a085"),]for x, y_pos, text, color in boxes:    rect = plt.Rectangle((x, y_pos-0.6), 1.8, 1.2, linewidth=2,                           edgecolor=color, facecolor=color, alpha=0.15)    ax.add_patch(rect)    rect2 = plt.Rectangle((x, y_pos-0.6), 1.8, 1.2, linewidth=2,                            edgecolor=color, facecolor='none')    ax.add_patch(rect2)    ax.text(x+0.9, y_pos, text, ha='center', va='center', fontsize=8, fontweight='bold')# Arrowsarrow_props = dict(arrowstyle='->', color='#333', lw=1.5)ax.annotate('', xy=(2.8, 3.5), xytext=(2.3, 3.5), arrowprops=arrow_props)ax.annotate('', xy=(5.3, 3.5), xytext=(4.6, 3.5), arrowprops=arrow_props)ax.annotate('', xy=(7.8, 3.5), xytext=(7.1, 3.5), arrowprops=arrow_props)ax.annotate('', xy=(10.0, 3.5), xytext=(9.6, 3.5), arrowprops=arrow_props)ax.annotate('', xy=(6.2, 2.2), xytext=(10.9, 2.9), arrowprops=arrow_props)ax.annotate('', xy=(8.5, 1.0), xytext=(7.1, 1.0), arrowprops=arrow_props)ax.annotate('', xy=(0.5, 3.5), xytext=(9.5, 0.4), arrowprops=dict(arrowstyle='->', color='#27ae60',            lw=1.5, connectionstyle="arc3,rad=0.3"))ax.text(4.5, 0.2, "Update pool & repeat weekly", fontsize=8, color='#27ae60', style='italic')plt.savefig("m4_active_learning_workflow.png", dpi=150, bbox_inches='tight')plt.close()print("[SAVED] m4_active_learning_workflow.png")## SECTION 3: INTEGRATION INTO COMPLETE AI SYSTEM (3.0 pts)#print("\n" + "="*70)print("SECTION 3: COMPLETE SYSTEM INTEGRATION")print("="*70)# ─── 3.1 Complete System Architecture Diagram ───fig, ax = plt.subplots(1, 1, figsize=(14, 10))ax.set_xlim(0, 14)ax.set_ylim(0, 10)ax.axis('off')ax.set_title("Adaptive Study Buddy — Complete System Architecture (M1–M4)",             fontsize=14, fontweight='bold', pad=15)# Main pipeline boxespipeline_boxes = [    (1, 9, 3, 0.7, "Student Interaction\n(Event Logs)", "#3498db"),    (5, 9, 3.5, 0.7, "Feature Engineering (M1)\n8 features + StandardScaler", "#2ecc71"),    (5, 7.8, 3.5, 0.7, "Pre-flight Checks (M3)\nSchema + Null + Range", "#f39c12"),    (5, 6.6, 3.5, 0.7, "MLP Model (M2)\n128→64→32→1 + T-scaling", "#e74c3c"),    (5, 5.4, 3.5, 0.7, "Post-Prediction Guards (M3)\nConfidence Flagging", "#9b59b6"),    (5, 4.2, 3.5, 0.7, "Recommendation Engine (M1)\nAdaptive Question Selection", "#1abc9c"),]for x, y_pos, w, h, text, color in pipeline_boxes:    rect = plt.Rectangle((x, y_pos), w, h, linewidth=2,                           edgecolor=color, facecolor=color, alpha=0.12)    ax.add_patch(rect)    rect2 = plt.Rectangle((x, y_pos), w, h, linewidth=2,                            edgecolor=color, facecolor='none')    ax.add_patch(rect2)    ax.text(x+w/2, y_pos+h/2, text, ha='center', va='center', fontsize=7.5, fontweight='bold')# Arrows (main pipeline)for i in range(len(pipeline_boxes)-1):    x1 = pipeline_boxes[i][0] + pipeline_boxes[i][2]/2    y1 = pipeline_boxes[i][1]    x2 = pipeline_boxes[i+1][0] + pipeline_boxes[i+1][2]/2    y2 = pipeline_boxes[i+1][1] + pipeline_boxes[i+1][3]    if i == 0:        ax.annotate('', xy=(5, 9.35), xytext=(4, 9.35), arrowprops=dict(arrowstyle='->', lw=1.5))    else:        ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='->', lw=1.2))# M4 Components (right side)m4_boxes = [    (10, 8.5, 3.5, 0.7, "PSI Drift Detection (M3)\nHourly on all features", "#e67e22"),    (10, 7.3, 3.5, 0.7, "🧑 HUMAN GATE (M4)\nApprove/Reject drift alert", "#c0392b"),    (10, 6.1, 3.5, 0.7, "Experience Replay (M4)\nReservoir Buffer (10K)", "#2c3e50"),    (10, 4.9, 3.5, 0.7, "🧑 HUMAN ANNOTATOR (M4)\nLabel uncertain samples", "#c0392b"),    (10, 3.7, 3.5, 0.7, "Active Learning Query (M4)\nHybrid Uncert.+Diversity", "#8e44ad"),    (10, 2.5, 3.5, 0.7, "Model Versioning (M2-M4)\nv4.x + Rollback", "#34495e"),]for x, y_pos, w, h, text, color in m4_boxes:    rect = plt.Rectangle((x, y_pos), w, h, linewidth=2,                           edgecolor=color, facecolor=color, alpha=0.12)    ax.add_patch(rect)    rect2 = plt.Rectangle((x, y_pos), w, h, linewidth=2,                            edgecolor=color, facecolor='none')    ax.add_patch(rect2)    ax.text(x+w/2, y_pos+h/2, text, ha='center', va='center', fontsize=7, fontweight='bold')# Connecting arrowsax.annotate('', xy=(10, 8.85), xytext=(8.5, 8.85),            arrowprops=dict(arrowstyle='->', lw=1.2, color='#e67e22'))ax.annotate('', xy=(10, 7.65), xytext=(10, 8.5),            arrowprops=dict(arrowstyle='->', lw=1.2))ax.annotate('', xy=(10, 6.45), xytext=(10, 7.3),            arrowprops=dict(arrowstyle='->', lw=1.2))ax.annotate('', xy=(10, 5.25), xytext=(10, 6.1),            arrowprops=dict(arrowstyle='->', lw=1.2))ax.annotate('', xy=(10, 4.05), xytext=(10, 4.9),            arrowprops=dict(arrowstyle='->', lw=1.2))ax.annotate('', xy=(10, 2.85), xytext=(10, 3.7),            arrowprops=dict(arrowstyle='->', lw=1.2))# Feedback loopax.annotate('', xy=(8.5, 6.95), xytext=(10, 2.85),            arrowprops=dict(arrowstyle='->', lw=1.5, color='#27ae60',                           connectionstyle="arc3,rad=-0.4"))ax.text(9, 1.8, "Deploy updated model", fontsize=8, color='#27ae60', style='italic', ha='center')# Labelsax.text(7, 10, "MONITORING & ADAPTATION LOOP (M3-M4)", fontsize=10,        fontweight='bold', ha='center', color='#2c3e50',        bbox=dict(boxstyle='round,pad=0.3', facecolor='#ecf0f1', edgecolor='#bdc3c7'))plt.savefig("m4_system_architecture.png", dpi=150, bbox_inches='tight')plt.close()print("[SAVED] m4_system_architecture.png")# ─── 3.2 Updated Failure Modes ───print("\n--- 3.2 Updated Failure Modes & Monitoring Plan ---")new_failures = [    ("Reservoir buffer contamination", "CL", "Validate buffer samples; quarterly refresh"),    ("Human annotator latency", "HITL", "48h SLA; fallback to self-labeling"),    ("Annotation quality degradation", "HITL", "Inter-annotator agreement monitoring"),    ("Update-inference race condition", "CL", "Blue-green deployment; atomic swap"),    ("Concept drift masking by CL", "CL", "Quarterly deep evaluation vs fresh retrain"),]print(f"\n{'Failure Mode':<35} {'Category':<10} {'Mitigation'}")print("-" * 90)for f, cat, mit in new_failures:    print(f"{f:<35} {cat:<10} {mit}")# ─── 3.3 Impact Analysis ───print("\n--- 3.3 CL & HITL Impact on System Properties ---")print("  Latency (inference):  UNCHANGED — buffer is training-time only")print(f"  Latency (update):     ~{np.mean([r['update_time'] for r in drift_results[1:]]):.0f}s per cycle")print(f"  Accuracy improvement: +{al_results[-1]['auroc'] - clean_metrics['AUROC']:.4f} AUROC (AL)")print(f"  Buffer overhead:      {buffer.memory_mb():.2f} MB")print(f"  Human time:           ~50 min/week (100 labels × 30s)")# ─── 3.4 Summary Table Across All Milestones ───print("\n--- 3.4 Cross-Milestone Summary ---")# Get final metricsfinal_metrics, _ = evaluate(model_al, X_test, y_test)final_latency = measure_latency(model_al, X_test)# Also evaluate on drifted dataX_drifted_t1 = generate_drifted_data(X_test, y_test, 1, train_mean, train_std)model_final = copy.deepcopy(model_al)metrics_drifted, _ = evaluate(model_final, X_drifted_t1, y_test)model_final, _ = incremental_finetune_er(    model_final,    generate_drifted_data(X_train[:5000], y_train[:5000], 1, train_mean, train_std),    y_train[:5000], buffer_al, X_val, y_val, CONFIG)metrics_after_adapt, _ = evaluate(model_final, X_drifted_t1, y_test)print(f"\n{'Metric':<25} {'M1':<12} {'M2':<12} {'M3':<12} {'M4':<12}")print("-" * 73)summary_data = [    ("Model", "LogReg", "MLP", "MLP+Jitter", "MLP+ER+AL"),    ("AUROC (clean)", "0.79*", "0.60**", f"{clean_metrics['AUROC']:.4f}", f"{final_metrics['AUROC']:.4f}"),    ("F1", "—", "—", f"{clean_metrics['F1']:.4f}", f"{final_metrics['F1']:.4f}"),    ("Precision", "—", "—", f"{clean_metrics['Precision']:.4f}", f"{final_metrics['Precision']:.4f}"),    ("Recall", "—", "—", f"{clean_metrics['Recall']:.4f}", f"{final_metrics['Recall']:.4f}"),    ("AUROC (drifted T=1)", "—", "—", "0.7690", f"{metrics_drifted['AUROC']:.4f}"),    ("AUROC (after adapt.)", "—", "—", "0.7922", f"{metrics_after_adapt['AUROC']:.4f}"),    ("Parameters", "9", "11,777", "11,905", f"{param_count:,}"),    ("Model Size (MB)", "<0.01", "0.047", "0.048", f"{model_size_mb:.4f}"),    ("p50 Latency (ms)", "—", "0.13", f"{baseline_latency['p50_ms']:.3f}", f"{final_latency['p50_ms']:.3f}"),    ("p90 Latency (ms)", "—", "0.22", f"{baseline_latency['p90_ms']:.3f}", f"{final_latency['p90_ms']:.3f}"),    ("Throughput (sps)", "—", "749K", f"{baseline_latency['throughput_sps']/1000:.0f}K", f"{final_latency['throughput_sps']/1000:.0f}K"),    ("Training Time (s)", "—", "87.8", f"{baseline_train_time:.1f}", f"{np.mean([r['update_time'] for r in drift_results[1:]]):.1f} (upd)"),]for row in summary_data:    print(f"{row[0]:<25} {row[1]:<12} {row[2]:<12} {row[3]:<12} {row[4]:<12}")# ─── Final Performance Table ───print("\n--- Final Performance Under Various Conditions ---")# Gaussian noiseX_noisy = X_test + np.random.normal(0, 0.1, X_test.shape).astype("float32")metrics_noise, _ = evaluate(model_al, X_noisy, y_test)# FGSMmodel_al.eval()X_test_t = torch.from_numpy(X_test).to(device).requires_grad_(True)y_test_t = torch.from_numpy(y_test).to(device)logits = model_al(X_test_t)loss_adv = nn.BCEWithLogitsLoss()(logits, y_test_t)loss_adv.backward()X_fgsm = (X_test_t + 0.1 * X_test_t.grad.sign()).detach().cpu().numpy()metrics_fgsm, _ = evaluate(model_al, X_fgsm, y_test)# Feature maskingX_masked = X_test.copy()X_masked[:, :2] = 0metrics_masked, _ = evaluate(model_al, X_masked, y_test)print(f"\n{'Condition':<30} {'AUROC':<10} {'F1':<10} {'Precision':<10} {'Recall':<10}")print("-" * 70)for name, m in [("Clean data", final_metrics), ("Gaussian noise (σ=0.1)", metrics_noise),                ("FGSM (ε=0.1)", metrics_fgsm), ("Feature masking (2 feat.)", metrics_masked),                ("Drifted (T=1, mild)", metrics_drifted), ("After CL+AL update", metrics_after_adapt)]:    print(f"{name:<30} {m['AUROC']:<10.4f} {m['F1']:<10.4f} {m['Precision']:<10.4f} {m['Recall']:<10.4f}")# ─── Save Model Artifacts ───print("\n--- Saving Artifacts ---")dataset_hash = hashlib.sha256(str(len(df)).encode()).hexdigest()[:12]artifact = {    "version": "v4.0",    "date": datetime.datetime.now().isoformat(),    "dataset_hash": dataset_hash,    "config": CONFIG,    "model_state_baseline": baseline_state,    "model_state_cl_updated": model_cl.state_dict(),    "model_state_al_updated": model_al.state_dict(),    "reservoir_buffer": {"X": buffer.X[:buffer.size], "y": buffer.y[:buffer.size]},    "scaler_mean": scaler.mean_.tolist(),    "scaler_scale": scaler.scale_.tolist(),    "features": FEATURES,}torch.save(artifact, "milestone4_artifacts.pt")print(f"  Saved milestone4_artifacts.pt (version v4.0, hash {dataset_hash})")# Save configwith open("model_config_v4.txt", "w") as f:    f.write(f"Version: v4.0\n")    f.write(f"Date: {datetime.datetime.now().isoformat()}\n")    f.write(f"Dataset hash: {dataset_hash}\n")    f.write(f"Features: {FEATURES}\n")    f.write(f"Config: {json.dumps(CONFIG, indent=2)}\n")    f.write(f"Scaler mean: {scaler.mean_.tolist()}\n")    f.write(f"Scaler scale: {scaler.scale_.tolist()}\n")    f.write(f"Final AUROC (clean): {final_metrics['AUROC']:.4f}\n")    f.write(f"Final F1 (clean): {final_metrics['F1']:.4f}\n")print("  Saved model_config_v4.txt")# ─── Summary Plot (4-panel) ───fig, axes = plt.subplots(2, 2, figsize=(14, 10))fig.suptitle("Milestone 4 — Complete Results Dashboard", fontsize=14, fontweight='bold')# Panel 1: CL trajectoriessteps_plot = list(range(6))axes[0,0].plot(steps_plot, auroc_before_list, 'r-o', label='Before ER Update', linewidth=2)axes[0,0].plot(steps_plot, auroc_after_list, 'g-s', label='After ER Update', linewidth=2)axes[0,0].axhline(y=clean_metrics["AUROC"], color='b', linestyle='--', alpha=0.5, label='Baseline')axes[0,0].set_xlabel("Drift Time Step")axes[0,0].set_ylabel("AUROC")axes[0,0].set_title("Continual Learning: Drift Recovery")axes[0,0].legend(fontsize=8)axes[0,0].grid(True, alpha=0.3)# Panel 2: Active Learning comparisonaxes[0,1].plot(cycles, al_aurocs, 'g-s', label='Hybrid', linewidth=2)axes[0,1].plot(cycles, unc_aurocs, 'b-^', label='Uncertainty', linewidth=2)axes[0,1].plot(cycles, rand_aurocs, 'r-o', label='Random', linewidth=2)axes[0,1].axhline(y=clean_metrics["AUROC"], color='gray', linestyle='--', alpha=0.5)axes[0,1].set_xlabel("AL Cycle")axes[0,1].set_ylabel("AUROC")axes[0,1].set_title("Active Learning: Strategy Comparison")axes[0,1].legend(fontsize=8)axes[0,1].grid(True, alpha=0.3)# Panel 3: Forgettingaxes[1,0].bar(steps_plot, forgetting_list, color=['#27ae60' if f < 0.005 else '#e67e22' for f in forgetting_list])axes[1,0].axhline(y=0.02, color='r', linestyle='--', label='Rollback threshold')axes[1,0].set_xlabel("Drift Time Step")axes[1,0].set_ylabel("Forgetting (ΔAUROC on clean)")axes[1,0].set_title("Catastrophic Forgetting")axes[1,0].legend(fontsize=8)axes[1,0].grid(True, alpha=0.3)# Panel 4: Performance comparison bar chartconditions = ['Clean', 'Noise\n(σ=0.1)', 'FGSM\n(ε=0.1)', 'Mask\n(2 feat)', 'Drifted\n(T=1)', 'After\nCL+AL']auroc_vals = [final_metrics['AUROC'], metrics_noise['AUROC'], metrics_fgsm['AUROC'],              metrics_masked['AUROC'], metrics_drifted['AUROC'], metrics_after_adapt['AUROC']]colors = ['#27ae60', '#3498db', '#e74c3c', '#e67e22', '#9b59b6', '#2ecc71']axes[1,1].bar(conditions, auroc_vals, color=colors, edgecolor='white', linewidth=1.5)axes[1,1].axhline(y=0.75, color='r', linestyle='--', alpha=0.5, label='SLO (0.75)')axes[1,1].set_ylabel("AUROC")axes[1,1].set_title("Final Performance Under Various Conditions")axes[1,1].legend(fontsize=8)axes[1,1].grid(True, alpha=0.3, axis='y')axes[1,1].set_ylim(0.5, 0.9)plt.tight_layout()plt.savefig("m4_results_dashboard.png", dpi=150, bbox_inches='tight')plt.close()print("[SAVED] m4_results_dashboard.png")# FINAL SUMMARYprint("\n" + "="*70)print("MILESTONE 4 COMPLETE — ALL REQUIREMENTS FULFILLED")print("="*70)print(f"""Deliverables:  1. Continual Learning (Experience Replay):     - 5-step drift simulation with metric trajectories     - Max forgetting: {max(forgetting_list):.4f} (below 0.02 threshold)     - Avg update time: {np.mean([r['update_time'] for r in drift_results[1:]]):.1f}s  2. Human-in-the-Loop & Active Learning:     - Hybrid uncertainty+diversity query strategy     - 5 AL cycles, 500 total labels (0.24% of training data)     - Final AUROC: {al_results[-1]['auroc']:.4f} (Δ = {al_results[-1]['auroc'] - clean_metrics['AUROC']:+.4f})     - {gain_hybrid/max(gain_random, 1e-6):.1f}× efficiency vs random sampling  3. System Integration:     - Complete system diagram saved     - Updated failure modes table     - Cross-milestone summary table     - Final performance under 6 conditions  Artifacts saved:     - milestone4_artifacts.pt (model states + buffer + config)     - model_config_v4.txt     - m4_continual_learning_trajectories.png     - m4_active_learning_comparison.png     - m4_active_learning_workflow.png     - m4_system_architecture.png     - m4_results_dashboard.png  GitHub: https://github.com/SohanaMahmud/adaptive_study_buddy_milestone_4""")